In [1]:
import json
import glob
import os
import re
import tol_colors as tc
import time

import numpy as np
import matplotlib.pyplot as plt

from qaoa_training_pipeline.evaluation import MPSEvaluator
from qiskit.quantum_info import SparsePauliOp
from qaoa_training_pipeline.utils.graph_utils import load_graph, graph_to_operator, solve_max_cut
from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import SwapStrategy
from qopt_best_practices.sat_mapping import SATMapper

/u/jjuradod/miniconda3/envs/cuquantum_env/lib/python3.11/site-packages/cupy/_environment.py:670: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy, cupy-cuda13x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [ ]:
def find_cycles_k(G, k):
    cycles = set()

    def dfs(path, start):
        if len(path) == k:
            if G.has_edge(path[-1], start):
                cycle = tuple(sorted(path))
                cycles.add(cycle)
            return

        for neighbor in G.neighbors(path[-1]):
            if neighbor not in path:
                dfs(path + [neighbor], start)

    for node in G.nodes():
        dfs([node], node)

    return cycles

In [ ]:
basepath  = "tests/"
glob_search = "*000N24L2S*MC_I_SV_*"

In [ ]:
def get_mps_evaluator(bond_dim, truncation_threshold=1.0e-10) -> MPSEvaluator:
    return MPSEvaluator(
        bond_dim_circuit=bond_dim, 
        threshold_circuit=truncation_threshold, 
        use_vidal_form=True, 
        use_swap_strategy=True,
        store_schmidt_values=True,
        store_intermediate_schmidt_values=True
    )

NameError: name 'MPSEvaluator' is not defined

In [ ]:
bond_dims = [2, 4, 8, 16, 32, 64]
swaps = [0, 2, 4, 8, 10, 12, 14, 16, 18]

In [ ]:
sv_energy_depth = {
    "2":{}, 
    "3":{}
}
mps_energy_depth = {
    "2":{
        
    }, 
    "3":{

    }
}
for swap in swaps: 
    mps_energy_depth["2"][swap] = []
    mps_energy_depth["3"][swap] = []
for file in glob.iglob(os.path.join(basepath, glob_search), recursive=True):
    match = re.search(r"S(\d+)", file)
    print(file, match)
    if match:
        d = int(match.group(1))
        if d not in swaps:
            continue
        num_swaps = d
    with open(file, "r") as f:
        data = json.load(f)
        for trainer in data.keys():
            if not trainer.isdigit() or trainer != "2":
                continue
            for depth in data[trainer].keys():
                if not depth.isdigit():
                    continue                
                sv_energy_depth[depth][num_swaps]=data[trainer][depth]["energy"]

                #Compute MPS energy
                params = data[trainer][depth]["optimized_qaoa_angles"]
                for bond_dim in bond_dims:
                    mps_evaluator = get_mps_evaluator(bond_dim=bond_dim)
                    mps_energy_depth[depth][num_swaps].append(mps_evaluator.evaluate(SparsePauliOp.from_list(data["cost_operator"]), params))

tests/20260601_105548_000N24L2S4_MC_I_SV_3.json <re.Match object; span=(30, 32), match='S4'>


/u/jjuradod/miniconda3/envs/cuquantum_env/lib/python3.11/site-packages/cotengra/hyperoptimizers/hyper.py:36: UserWarning: Couldn't import `kahypar` - skipping from default hyper optimizer and using basic `labels` method instead. `kahypar` is highly recommended for the best quality contraction paths.
  warnings.warn(
/u/jjuradod/miniconda3/envs/cuquantum_env/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


tests/20260601_053022_000N24L2S0_MC_I_SV_1.json <re.Match object; span=(30, 32), match='S0'>
tests/20260601_111052_000N24L2S8_MC_I_SV_3.json <re.Match object; span=(30, 32), match='S8'>
tests/20260601_095109_000N24L2S20_MC_I_SV_3.json <re.Match object; span=(30, 33), match='S20'>
tests/20260601_104218_000N24L2S2_MC_I_SV_3.json <re.Match object; span=(30, 32), match='S2'>
tests/20260601_073019_000N24L2S10_MC_I_SV_3.json <re.Match object; span=(30, 33), match='S10'>
tests/20260601_075010_000N24L2S12_MC_I_SV_3.json <re.Match object; span=(30, 33), match='S12'>
tests/20260601_084411_000N24L2S16_MC_I_SV_3.json <re.Match object; span=(30, 33), match='S16'>
tests/20260601_081436_000N24L2S14_MC_I_SV_3.json <re.Match object; span=(30, 33), match='S14'>
tests/20260601_091917_000N24L2S18_MC_I_SV_3.json <re.Match object; span=(30, 33), match='S18'>


In [ ]:
depths = sv_energy_depth.keys()
energies = [np.mean(list(values.values())) for values in sv_energy_depth.values()]
plt.figure(figsize=(8, 5))
plt.plot(depths, energies, marker="o")
plt.xlabel("Depth")
plt.ylabel("Energy")
plt.title("Mean energy vs depth")
plt.grid(True)

For depth 3 I inspect how the MPS energy varies in terms of bond dimension and swap layers

In [ ]:
sv_energy = sv_energy_depth["3"]
mps_energy = mps_energy_depth["3"]


cmap = tc.rainbow_discrete(n_colors=10)
colors = cmap(np.linspace(0, 1, 10))

plt.figure(figsize=(8, 5))
for s, swap in enumerate(mps_energy.keys()):
    plt.plot(bond_dims, mps_energy[swap], label=f"# of swaps: {swap}", marker="o", color=colors[s])

    plt.hlines(sv_energy[swap], 0, 64, color=colors[s], linestyle='--')
plt.xlabel("Bond dimension")
plt.ylabel("Energy")
plt.title("Energy vs bond dimension")
plt.grid(True)
plt.legend()

Profiling of GPU implementation against CPU for 20 nodes line-based graphs

In [ ]:
results_path = "QAOA-Parameter-Setting/data/training/line_to_full/"
instances_glob = "*000N20*MC*I_SV*_10.json"
# instances_glob = "*001N100*MC*I_MPSAer_optBD24*_10.json"

In [ ]:
bond_dims = [2, 8, 32, 64] 

In [ ]:
from qaoa_training_pipeline.evaluation.cuquantum_mps import CuQuantumMPSEvaluator
def get_cuquantum_evaluator(bond):
    cuQuantum_evaluator = CuQuantumMPSEvaluator(
        max_bond_dim=bond, 
        rel_cutoff=1e-10, 
        abs_cutoff=None, 
        precision="fp64", 
        mpo_application="approximate", 
        gauge_option="simple", 
        normalization="L2", 
        svd_algo="gesvd", 
        use_swap_strategy=True
    )

    return cuQuantum_evaluator

In [ ]:
from collections import defaultdict
energy_cpu = defaultdict(list)
energy_gpu =  defaultdict(list)
times_cpu =  defaultdict(list)
times_gpu =  defaultdict(list)

sv_energy = {}
for file in glob.iglob(os.path.join(results_path, instances_glob)):
    with open(file, "r") as f:
        data = json.load(f)
        match = re.search(r"S(\d+)", file)
        print(file, match)
        if match:
            d = int(match.group(1))
            num_swaps = d
        sv_energy[num_swaps] = data["2"]["energy"]
        params = data["2"]["optimized_qaoa_angles"]

        graph = load_graph(os.path.join("QAOA-Parameter-Setting/instances/line_to_full",data["args"]["input"].split("/")[-1]))
        cost_op = graph_to_operator(graph, pre_factor=-0.5)
        swap_strategy = SwapStrategy.from_line(range(graph.order()))
        start = time.time()
        sat_graph, _, layers = SATMapper(timeout=10).remap_graph_with_sat(graph=graph, swap_strategy=swap_strategy)
        sat_cost_op = graph_to_operator(sat_graph, pre_factor=-0.5)
        print(f"Mapped graph in {time.time() - start} sec")

        

        for bond in bond_dims:
            cpu_evaluator = get_mps_evaluator(bond)
            t1 = time.time()
            e_cpu = cpu_evaluator.evaluate(sat_cost_op, params)
            t2 = time.time()
            energy_cpu[num_swaps].append(e_cpu)
            times_cpu[num_swaps].append(t2 - t1)
            
            gpu_evaluator = get_cuquantum_evaluator(bond)
            t1 = time.time()
            e_gpu = gpu_evaluator.evaluate(sat_cost_op, params)
            t2 = time.time()
            energy_gpu[num_swaps].append(e_gpu)
            times_gpu[num_swaps].append(t2 - t1)

In [ ]:
from matplotlib.lines import Line2D
# colors = list(tc.bright)
cmap = tc.rainbow_discrete(n_colors=10)
colors = cmap(np.linspace(0, 1, 10))
fig, axs = plt.subplots(1, 2, figsize=(18, 5))

for s, swap in enumerate(energy_cpu.keys()):
    axs[0].plot(bond_dims, energy_gpu[swap], label=f"# of swaps: {swap}", marker="o", color=colors[s])
    axs[0].plot(bond_dims, energy_cpu[swap], label=f"# of swaps: {swap}", marker="^", color=colors[s])

    axs[0].hlines(sv_energy[swap], bond_dims[0], bond_dims[-1], color=colors[s], linestyle='--')

    axs[1].plot(bond_dims, times_gpu[swap], label=f"# of swaps: {swap}", marker="o", color=colors[s])
    axs[1].plot(bond_dims, times_cpu[swap], label=f"# of swaps: {swap}", marker="^", color=colors[s])
    

axs[0].set_xlabel("Bond dimension")
axs[0].set_ylabel("Energy")
axs[1].set_xlabel("Bond dimension")
axs[1].set_ylabel("Evaluation time (s)")
axs[1].set_yscale('log')
axs[1].set_xscale('log')
axs[0].grid(True)

marker_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor="black",
        markeredgecolor="black",
        markersize=8,
        label="GPU"
    ),
    Line2D(
        [0], [0],
        marker="^",
        color="none",
        markerfacecolor="black",
        markeredgecolor="black",
        markersize=8,
        label="CPU"
    ),
    Line2D(
        [0], [0],
        linestyle="--",
        color="black",
        label="Statevector"
    ),
]
line_handles = [
    Line2D(
        [0], [0],
        color=colors[s],
        label=f"{swaps}"
    ) for s, swaps in enumerate(energy_cpu.keys())
]

leg1 = fig.legend(
    handles=marker_handles,
    bbox_to_anchor=(0.7,0),
    title="Evaluators",
    ncol=3
)

fig.add_artist(leg1)

leg2 = fig.legend(
    handles=line_handles,
    bbox_to_anchor=(0.5, 0),
    title="Num SWAPS",
    ncol= 6
)


Analysis of the effect of system size and number of SWAPs on the error for p=1

In [ ]:
graph_type = "erdos_renyi"

instances_path = f"QAOA-Parameter-Setting/instances/{graph_type}/"

In [ ]:
import os
import re

# pattern = re.compile(r'000_(1[5-9]|[2-3][0-9])nodes_([0-9])swap_layers\.json$')
# # pattern = re.compile(r'*nodes_*swap_layers\.json$')

# matched = [
#     os.path.join(instances_path, f) for f in os.listdir(instances_path)
#     if pattern.match(f)
# ]
# print(len(matched))
# matched

matched = [
    os.path.join(instances_path, f) for f in os.listdir(instances_path)
]

len(matched)

In [ ]:
#Load best parameters summary
from qaoa_parameter_setting.utils.best_parameters import BestParameterManager

best_param_manager = BestParameterManager()

best_param_manager.add_data(f"QAOA-Parameter-Setting/data/training/{graph_type}/")

best_params = best_param_manager.data

In [ ]:
bond_dims = [2, 4, 6, 8, 10, 12, 16, 24, 32, 54, 64, 128, 256, 512]
depth = "3"

In [ ]:
from qaoa_training_pipeline.utils.graph_utils import load_graph, graph_to_operator, solve_max_cut
from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import SwapStrategy
from qaoa_training_pipeline.training import DepthOneScanTrainer
from qaoa_training_pipeline.qaoa_training_pipeline.evaluation import EfficientDepthOneEvaluator
from qopt_best_practices.sat_mapping import SATMapper

import networkx as nx
from collections import defaultdict


results_inst = defaultdict(dict)
bond_dims = [2, 4, 6, 8, 10, 12, 16, 24, 32, 54, 64, 128, 256, 512]
depth = "3"
results_inst["settings"]["bond_dims"] = bond_dims
results_inst["settings"]["depth"] = depth

#Open already produced results
# with open("evaluation_profiling/results_line_to_full_3.json", 'r') as f:
#     results_inst = json.load(f)

for file in matched:
    instance_name = file.split("/")[-1]
    instance_minmax =instance_name.split(".")[0] + "_maxmin_cut.json"
    if instance_name in best_params["SV"].keys():
        params = best_params["SV"][instance_name][depth]['qaoa_angles']
    elif instance_name in best_params["PP"].keys():
        params = best_params["PP"][instance_name][depth]['qaoa_angles']
    elif instance_name in best_params["MPS"].keys():
        params = best_params["MPS"][instance_name][depth]['qaoa_angles']
    else:
        print("Best results not found")
        continue
    swaps = re.search(r"(\d+)swap", file)
    print(file, swaps)
    # if swaps:
    #     d = int(swaps.group(1))
    #     num_swaps = d
    nodes = re.search(r"(\d+)nodes", file)
    if nodes:
        d = int(nodes.group(1))
        num_nodes = d
    index = re.search(r"^(\d{3})_.*", file)
    if index:
        d = int(index.group(1))
        num_index = d


    graph = load_graph(file)
    cost_op = graph_to_operator(graph, pre_factor=-0.5)
    swap_strategy = SwapStrategy.from_line(range(graph.order()))
    start = time.time()
    sat_graph, _, layers = SATMapper(timeout=10).remap_graph_with_sat(graph=graph, swap_strategy=swap_strategy)
    print(f"Mapped graph in {time.time() - start} sec")


    try:
        with open(f"QAOA-Parameter-Setting/data/minmax_cuts/{graph_type}/{instance_minmax}", 'r') as f:
            minmax_cuts = json.load(f)
            energy_opt = minmax_cuts["max_cut"] - minmax_cuts["sum_of_weights"]/2
    except:
        max_cut, _, _ = solve_max_cut(cost_op)
        energy_opt = max_cut - sum(data["weight"] for u, v, data in graph.edges(data=True))



    for i, bond in enumerate(bond_dims):
        cuquantum_evaluator = get_cuquantum_evaluator(bond)
        
        t1 = time.time()
        mps_energy = np.abs(cuquantum_evaluator.evaluate(graph_to_operator(sat_graph, pre_factor=-0.5), params))
        t2 = time.time()
        if i == 0:
            results_inst[file]["mps_energy"] = [mps_energy]
            results_inst[file]["mps_energy_times"] = [t2-t1]
        else:
            results_inst[file]["mps_energy"].append(mps_energy)
            results_inst[file]["mps_energy_times"].append(t2-t1)
    results_inst[file]["energy"] = energy_opt
    results_inst[file]["angles"] = params
    #results_inst[file]["swaps"] = num_swaps
    results_inst[file]["density"] = nx.density(graph)
    # results_inst[file]["graph"] = graph
    with open(f"evaluation_profiling/results_{graph_type}_3.json", 'w') as f:
        json.dump(results_inst, f, indent=4)


In [ ]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

with open(f"evaluation_profiling/results{timestamp}_d{depth}.json", 'w') as f:
    json.dump(results_inst, f, indent=4)

In [ ]:
with open(f"evaluation_profiling/results_line_to_full_3.json", 'r') as f:
    results_inst = json.load(f)

In [ ]:
#Uniformly sample densities
densities_num = []
density_to_idx = {}
i=0
for result_file in results_inst.keys():
    if result_file == "settings":
        continue
    density = results_inst[result_file]["density"]
    if density in density_to_idx.keys():
        densities_num[density_to_idx[density]] +=1
    else:
        density_to_idx[density] = i
        densities_num.append(1)
        i+=1

In [ ]:
print(densities_num)
print(np.sort(list(density_to_idx.keys())))

In [ ]:
from matplotlib.lines import Line2D
# colors = list(tc.bright)
# fig, axs = plt.subplots(1, 2, figsize=(18, 5))

errors = []
densities = []
num_densities = []
density_to_idx = {}
i = 0

times_by_density = []

for result_file in results_inst.keys():
    if result_file == "settings":
        continue
    density = results_inst[result_file]["density"]
    t = np.array(results_inst[result_file]["mps_energy_times"])
    if density not in density_to_idx.keys():
        density_to_idx[density] = i
        errors.append(np.abs(np.array(results_inst[result_file]["mps_energy"]) - results_inst[result_file]["energy"])/results_inst[result_file]["energy"])
        times_by_density.append(t)
        densities.append(density)
        num_densities.append(1)
        i+=1
    else:
        errors[density_to_idx[density]] += np.abs(np.array(results_inst[result_file]["mps_energy"]) - results_inst[result_file]["energy"])/(results_inst[result_file]["energy"])
        
        times_by_density[density_to_idx[density]] += t
        num_densities[density_to_idx[density]] += 1
    

for density_idx in density_to_idx.keys():
    idx = density_to_idx[density_idx]
    N = num_densities[idx]
    errors[idx] /= N
    times_by_density[idx] /= N

densities_idxs = np.argsort(densities)
densities = np.array(densities)[densities_idxs]
X, Y = np.meshgrid(bond_dims, densities)
errors = np.array(errors, dtype=float )
ordered_errors = errors[densities_idxs,:]
ordered_times = np.array(times_by_density, dtype=float)[densities_idxs, :]

fig, axs = plt.subplots(1, 2, figsize=(18, 5))
im = axs[0].pcolormesh(X, Y, ordered_errors, cmap='viridis', shading='gouraud')
fig.colorbar(im, ax=axs[0])

axs[0].set_xscale("log")

im = axs[1].pcolormesh(X, Y, ordered_times, cmap='viridis')
fig.colorbar(im, ax=axs[1])

axs[1].set_xscale("log")



In [ ]:
np.max(errors)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 5))

sc = axs[0].scatter(X.flatten(), Y.flatten(),c= ordered_errors.flatten(), cmap='viridis')
fig.colorbar(sc, ax=axs[0])

axs[0].set_xscale("log")
axs[1].set_xscale("log")
sc = axs[1].scatter(X.flatten(), Y.flatten(),c= ordered_times.flatten(), cmap='viridis')
fig.colorbar(sc, ax=axs[1])


